# 2. Logistic Regression

**Machine Learning Fundamentals and Predictive Analytics — Notebook 2 of 11**

Linear regression predicts a number. **Logistic regression** predicts a **probability** — and
therefore a category. Will this customer churn? Is this email spam? Is this tumour malignant?

Despite the name it is a **classification** algorithm. It is the workhorse of applied
classification: fast, calibrated, interpretable, and the model that regulators and clinicians
actually accept.

### What you will learn

1. Why linear regression fails for classification
2. The **sigmoid** function and the **log-odds** view
3. **Log loss** (cross-entropy) and how the model is fitted
4. Predicting classes vs predicting probabilities, and the **decision threshold**
5. **Confusion matrix**, precision, recall, F1 — and choosing between them
6. **ROC-AUC** and **PR-AUC**, and when each is the right curve
7. Interpreting coefficients as **odds ratios**
8. **Regularisation** and the `C` parameter
9. **Class imbalance** and what to do about it
10. **Multiclass** logistic regression
11. **Calibration** — are the probabilities honest?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import (train_test_split, cross_val_score, StratifiedKFold,
                                     GridSearchCV)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, classification_report,
                             accuracy_score, precision_score, recall_score, f1_score,
                             roc_curve, roc_auc_score, precision_recall_curve,
                             average_precision_score, log_loss, brier_score_loss)
from sklearn.dummy import DummyClassifier

rng = np.random.default_rng(seed=2)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)
SKF = StratifiedKFold(5, shuffle=True, random_state=0)

---
## 2.1 Why not linear regression?

Suppose we code the outcome as 0/1 and fit a straight line. Three things go wrong:

1. Predictions go **below 0 and above 1** — meaningless as probabilities
2. The **fit is dragged** by extreme $x$ values, moving the implied boundary
3. The residuals are structurally non-Normal and heteroscedastic, so the inference is invalid

We need a function that maps any real number into $(0, 1)$. That function is the **sigmoid**
(or logistic):

$$\sigma(z) = \frac{1}{1 + e^{-z}}, \qquad z = \beta_0 + \beta_1 x_1 + \dots + \beta_p x_p$$

$$P(y = 1 \mid \mathbf{x}) = \sigma(\mathbf{x}^\top\boldsymbol\beta)$$

In [ ]:
# Exam pass/fail from hours studied
hours = np.r_[rng.uniform(0, 6, 40), rng.uniform(4, 14, 40)]
prob_pass = 1 / (1 + np.exp(-(hours - 6.5)))
passed = (rng.random(len(hours)) < prob_pass).astype(int)

Xh = hours.reshape(-1, 1)
lin = LinearRegression().fit(Xh, passed)
log = LogisticRegression().fit(Xh, passed)

grid = np.linspace(-2, 18, 400).reshape(-1, 1)
plt.scatter(hours, passed, s=45, c=passed, cmap="coolwarm", edgecolor="k", linewidth=0.4,
            zorder=3, label="observations (0 = fail, 1 = pass)")
plt.plot(grid, lin.predict(grid), color="darkorange", lw=2, label="linear regression")
plt.plot(grid, log.predict_proba(grid)[:, 1], color="steelblue", lw=2.4,
         label="logistic regression")
plt.axhline(0, color="grey", lw=0.8); plt.axhline(1, color="grey", lw=0.8)
plt.axhline(0.5, color="black", ls=":", lw=1)
plt.xlabel("hours studied"); plt.ylabel("P(pass)")
plt.title("The straight line leaves the [0,1] box; the sigmoid never does")
plt.legend(fontsize=8); plt.show()

print(f"Linear prediction at 0 hours  : {lin.predict([[0]])[0]:+.3f}  <- negative probability")
print(f"Linear prediction at 18 hours : {lin.predict([[18]])[0]:+.3f}  <- above 1")
print(f"Logistic at 0 and 18 hours    : {log.predict_proba([[0]])[0,1]:.4f}, "
      f"{log.predict_proba([[18]])[0,1]:.4f}")

In [ ]:
# The sigmoid, and what its parameters do
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

zs = np.linspace(-8, 8, 400)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(zs, sigmoid(zs), color="steelblue", lw=2.4)
ax[0].axhline(0.5, color="crimson", ls="--"); ax[0].axvline(0, color="crimson", ls="--")
ax[0].set_xlabel("z (the linear part)"); ax[0].set_ylabel("sigma(z)")
ax[0].set_title("sigma(0) = 0.5; saturates at 0 and 1")

for b1, colour in [(0.4, "steelblue"), (1.0, "seagreen"), (3.0, "crimson")]:
    ax[1].plot(zs, sigmoid(b1 * zs), color=colour, lw=2, label=f"slope = {b1}")
ax[1].set_xlabel("x"); ax[1].set_title("A larger coefficient = a sharper transition")
ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

print("Useful facts:")
print(f"  sigma(0)  = {sigmoid(0):.4f}")
print(f"  sigma(2)  = {sigmoid(2):.4f}      sigma(-2) = {sigmoid(-2):.4f}")
print(f"  sigma(z) + sigma(-z) = 1 always")
print(f"  derivative: sigma'(z) = sigma(z)(1 - sigma(z)), maximal at z = 0")

---
## 2.2 The log-odds view

Rearranging the sigmoid gives the identity that makes logistic regression interpretable:

$$\log\underbrace{\frac{p}{1-p}}_{\text{odds}} = \beta_0 + \beta_1 x_1 + \dots + \beta_p x_p$$

So logistic regression **is** a linear model — linear in the **log-odds** (also called the
*logit*). That is what "generalised linear model" means: a linear predictor passed through a
link function.

| $p$ | odds $p/(1-p)$ | log-odds |
|---|---|---|
| 0.10 | 0.11 | −2.20 |
| 0.25 | 0.33 | −1.10 |
| 0.50 | 1.00 | 0.00 |
| 0.75 | 3.00 | +1.10 |
| 0.90 | 9.00 | +2.20 |

The immediate consequence: a one-unit increase in $x_j$ **multiplies the odds** by
$e^{\beta_j}$. That factor is the **odds ratio**, and it is how logistic regression results
are reported in medicine and social science.

In [ ]:
ps = np.array([0.01, 0.1, 0.25, 0.5, 0.75, 0.9, 0.99])
print(f"{'p':>7}{'odds':>10}{'log-odds':>12}")
for p_ in ps:
    print(f"{p_:>7.2f}{p_/(1-p_):>10.3f}{np.log(p_/(1-p_)):>12.4f}")

b0, b1 = log.intercept_[0], log.coef_[0][0]
print(f"\nFitted model: log-odds(pass) = {b0:.3f} + {b1:.3f} * hours")
print(f"Odds ratio per extra hour = exp({b1:.3f}) = {np.exp(b1):.3f}")
print(f"  -> each extra hour of study multiplies the odds of passing by {np.exp(b1):.2f}")
print(f"Decision boundary (p = 0.5) is where z = 0: hours = {-b0/b1:.2f}")

---
## 2.3 How it is fitted: log loss

There is no closed-form solution. Instead we maximise the likelihood of the observed labels,
which is equivalent to minimising the **log loss** (binary cross-entropy):

$$J(\boldsymbol\beta) = -\frac{1}{n}\sum_{i=1}^{n}\Big[y_i\log \hat{p}_i + (1-y_i)\log(1-\hat{p}_i)\Big]$$

The shape of this loss is the point. For a positive example ($y=1$):

- predict $\hat{p} = 0.9$ → loss $= -\log(0.9) = 0.105$ (small)
- predict $\hat{p} = 0.5$ → loss $= 0.693$
- predict $\hat{p} = 0.01$ → loss $= 4.6$ (**huge**)

Log loss punishes **confident mistakes** brutally, which is exactly what you want from a model
whose output is a probability. It is convex, so gradient descent finds the global optimum.

In [ ]:
p_hat = np.linspace(0.001, 0.999, 500)
plt.plot(p_hat, -np.log(p_hat), color="steelblue", lw=2.2, label="true label = 1")
plt.plot(p_hat, -np.log(1 - p_hat), color="crimson", lw=2.2, label="true label = 0")
plt.ylim(0, 6); plt.xlabel("predicted probability of class 1"); plt.ylabel("log loss")
plt.title("Confident and wrong is catastrophically expensive")
plt.legend(fontsize=8); plt.show()

print("Loss for a positive example at various predictions:")
for p_ in (0.99, 0.9, 0.7, 0.5, 0.2, 0.01):
    print(f"  predicted {p_:.2f} -> loss {-np.log(p_):.4f}")

# Gradient descent on log loss, by hand
def fit_logistic_gd(X, y, lr=0.5, n_iter=3000):
    '''Logistic regression by batch gradient descent. Returns weights and loss history.'''
    Xb = np.column_stack([np.ones(len(X)), X])
    w = np.zeros(Xb.shape[1])
    hist = []
    for _ in range(n_iter):
        p = 1 / (1 + np.exp(-(Xb @ w)))
        w += lr * Xb.T @ (y - p) / len(y)               # gradient ascent on log-likelihood
        p = np.clip(1 / (1 + np.exp(-(Xb @ w))), 1e-12, 1 - 1e-12)
        hist.append(-(y*np.log(p) + (1-y)*np.log(1-p)).mean())
    return w, hist

w_gd, hist_gd = fit_logistic_gd(Xh, passed)
print(f"\nBy gradient descent : intercept {w_gd[0]:.4f}, coefficient {w_gd[1]:.4f}")
print(f"By scikit-learn     : intercept {log.intercept_[0]:.4f}, "
      f"coefficient {log.coef_[0][0]:.4f}")
print("(they differ slightly because sklearn regularises by default -- see section 2.8)")

plt.plot(hist_gd, color="steelblue")
plt.xlabel("iteration"); plt.ylabel("log loss"); plt.title("Log loss is convex: one minimum")
plt.show()

---
## 2.4 A realistic example: customer churn

From here on we work with a mixed-type dataset and the full evaluation toolkit.

In [ ]:
m = 3_000
tenure = rng.exponential(24, m).clip(0, 84)
monthly = rng.normal(65, 22, m).clip(15, 140)
contract = rng.choice(["monthly", "one_year", "two_year"], m, p=[0.55, 0.28, 0.17])
support_calls = rng.poisson(1.2, m)
has_fiber = rng.integers(0, 2, m)

contract_effect = {"monthly": 1.3, "one_year": -0.4, "two_year": -1.4}
z = (-1.4
     - 0.045 * tenure
     + 0.016 * monthly
     + np.array([contract_effect[c] for c in contract])
     + 0.30 * support_calls
     + 0.35 * has_fiber)
churn = (rng.random(m) < 1/(1+np.exp(-z))).astype(int)

df = pd.DataFrame({"tenure": tenure, "monthly": monthly, "contract": contract,
                   "support_calls": support_calls, "has_fiber": has_fiber, "churn": churn})
print(df.head())
print(f"\nChurn rate: {churn.mean():.3f}  ({churn.sum()} of {m})")
print("\nChurn rate by contract type:")
print(df.groupby("contract")["churn"].agg(["count", "mean"]).round(3).to_string())

In [ ]:
num_cols = ["tenure", "monthly", "support_calls", "has_fiber"]
cat_cols = ["contract"]

pipe = Pipeline([
    ("prep", ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols)])),
    ("model", LogisticRegression(max_iter=1000)),
])

X_all, y_all = df.drop(columns="churn"), df["churn"]
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.25,
                                          random_state=0, stratify=y_all)
pipe.fit(X_tr, y_tr)

proba = pipe.predict_proba(X_te)[:, 1]
pred = pipe.predict(X_te)

print(f"Cross-validated accuracy : "
      f"{cross_val_score(pipe, X_tr, y_tr, cv=SKF).mean():.4f}")
print(f"Test accuracy            : {accuracy_score(y_te, pred):.4f}")
print(f"Baseline (majority class): "
      f"{DummyClassifier(strategy='most_frequent').fit(X_tr, y_tr).score(X_te, y_te):.4f}")
print(f"Test log loss            : {log_loss(y_te, proba):.4f}")
print(f"Test ROC-AUC             : {roc_auc_score(y_te, proba):.4f}")

---
## 2.5 The confusion matrix and the metrics that come from it

|  | Predicted 0 | Predicted 1 |
|---|---|---|
| **Actual 0** | TN (true negative) | FP (false positive) — *false alarm* |
| **Actual 1** | FN (false negative) — *miss* | TP (true positive) |

$$\text{Accuracy} = \frac{TP+TN}{\text{all}} \qquad
\textbf{Precision} = \frac{TP}{TP+FP} \qquad
\textbf{Recall} = \frac{TP}{TP+FN}$$

$$F_1 = 2\cdot\frac{\text{precision}\cdot\text{recall}}{\text{precision}+\text{recall}}
\qquad \text{Specificity} = \frac{TN}{TN+FP}$$

**How to choose:**

| Question | Metric |
|---|---|
| Of those we flagged, how many were real? | **Precision** — use when false alarms are costly (spam folder, ad spend) |
| Of all the real cases, how many did we catch? | **Recall** — use when misses are costly (cancer screening, fraud) |
| Need one number balancing both | **F1** |
| Classes are balanced and errors cost the same | Accuracy |
| Need a threshold-free summary | **ROC-AUC** or **PR-AUC** |

Precision and recall trade off against each other: you cannot raise both by moving the
threshold. Only a better model raises both.

In [ ]:
cm = confusion_matrix(y_te, pred)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ConfusionMatrixDisplay(cm, display_labels=["stay", "churn"]).plot(ax=ax[0], cmap="Blues",
                                                                 colorbar=False)
ax[0].set_title("Counts")
ConfusionMatrixDisplay(confusion_matrix(y_te, pred, normalize="true"),
                       display_labels=["stay", "churn"]).plot(ax=ax[1], cmap="Blues",
                                                              colorbar=False,
                                                              values_format=".3f")
ax[1].set_title("Normalised by true class (row-wise)")
plt.tight_layout(); plt.show()

print(f"TN = {tn}   FP = {fp}   FN = {fn}   TP = {tp}\n")
print(f"Accuracy    = (TP+TN)/n = {(tp+tn)/cm.sum():.4f}")
print(f"Precision   = TP/(TP+FP) = {tp}/{tp+fp} = {tp/(tp+fp):.4f}")
print(f"Recall      = TP/(TP+FN) = {tp}/{tp+fn} = {tp/(tp+fn):.4f}")
print(f"Specificity = TN/(TN+FP) = {tn/(tn+fp):.4f}")
print(f"F1          = {f1_score(y_te, pred):.4f}")
print()
print(classification_report(y_te, pred, target_names=["stay", "churn"]))

---
## 2.6 The decision threshold is a business choice

`predict()` uses 0.5. **That default is arbitrary** and almost never optimal. The model gives
you a probability; converting it into an action is a decision that depends on costs.

- Retention team can call 200 customers a month → pick the threshold that flags 200
- A missed churn costs ₹8,000 in lifetime value; an unnecessary retention offer costs ₹500 →
  minimise expected cost
- Legal requirement to catch 95% of cases → pick the threshold that gives recall 0.95

Tuning the threshold is free. Do it before reaching for a fancier model.

In [ ]:
thresholds = np.linspace(0.05, 0.95, 91)
rows = []
for t in thresholds:
    p = (proba >= t).astype(int)
    rows.append({"threshold": t,
                 "precision": precision_score(y_te, p, zero_division=0),
                 "recall": recall_score(y_te, p, zero_division=0),
                 "f1": f1_score(y_te, p, zero_division=0),
                 "flagged": p.sum()})
tr_df = pd.DataFrame(rows)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(tr_df.threshold, tr_df.precision, color="steelblue", lw=2, label="precision")
ax[0].plot(tr_df.threshold, tr_df.recall, color="crimson", lw=2, label="recall")
ax[0].plot(tr_df.threshold, tr_df.f1, color="seagreen", lw=2, label="F1")
best_f1 = tr_df.loc[tr_df.f1.idxmax()]
ax[0].axvline(best_f1.threshold, color="black", ls="--",
              label=f"best F1 at {best_f1.threshold:.2f}")
ax[0].axvline(0.5, color="grey", ls=":", label="default 0.5")
ax[0].set_xlabel("threshold"); ax[0].legend(fontsize=8)
ax[0].set_title("Precision and recall pull in opposite directions")

ax[1].plot(tr_df.recall, tr_df.precision, color="steelblue", lw=2)
ax[1].set_xlabel("recall"); ax[1].set_ylabel("precision")
ax[1].set_title("The same information as a precision-recall curve")
plt.tight_layout(); plt.show()

print(f"Default threshold 0.50 : precision {tr_df.loc[tr_df.threshold.sub(0.5).abs().idxmin(),'precision']:.3f}, "
      f"recall {tr_df.loc[tr_df.threshold.sub(0.5).abs().idxmin(),'recall']:.3f}")
print(f"Best-F1 threshold {best_f1.threshold:.2f} : precision {best_f1.precision:.3f}, "
      f"recall {best_f1.recall:.3f}, F1 {best_f1.f1:.3f}")

In [ ]:
# Choosing the threshold by expected COST -- the version a business would sign off
COST_MISS = 8000.0        # a churned customer we failed to flag
COST_FALSE_ALARM = 500.0  # a retention offer to someone who was staying anyway

costs = []
for t in thresholds:
    p = (proba >= t).astype(int)
    tn_, fp_, fn_, tp_ = confusion_matrix(y_te, p).ravel()
    costs.append(fn_ * COST_MISS + fp_ * COST_FALSE_ALARM)
costs = np.array(costs)
t_best = thresholds[int(np.argmin(costs))]

plt.plot(thresholds, costs, color="steelblue", lw=2)
plt.axvline(t_best, color="crimson", ls="--", label=f"cost-optimal threshold = {t_best:.2f}")
plt.axvline(0.5, color="grey", ls=":", label="default 0.5")
plt.xlabel("threshold"); plt.ylabel("expected total cost on the test set")
plt.title("The threshold that minimises money, not F1")
plt.legend(fontsize=8); plt.show()

cost_default = costs[int(np.argmin(np.abs(thresholds - 0.5)))]
print(f"Cost at threshold 0.50 : {cost_default:,.0f}")
print(f"Cost at threshold {t_best:.2f} : {costs.min():,.0f}")
print(f"Saving from moving the threshold alone: {cost_default - costs.min():,.0f} "
      f"({(1 - costs.min()/cost_default)*100:.1f}%)")
print("\nNote the direction: because a miss costs 16x a false alarm, the optimal threshold")
print("is far BELOW 0.5 -- we deliberately accept many false alarms to avoid misses.")

---
## 2.7 ROC and PR curves

**ROC curve** — true positive rate (recall) against false positive rate, across all
thresholds. **ROC-AUC** is the area beneath it, and has a lovely interpretation: *the
probability that the model scores a random positive higher than a random negative.*
0.5 = coin flip, 1.0 = perfect.

**Precision-recall curve** — precision against recall. **PR-AUC** (average precision) is its
summary.

**Which to use?** ROC-AUC is insensitive to class balance, which sounds good but hides a
problem: with 1% positives, a huge number of false positives barely moves the false positive
rate, so ROC-AUC stays flattering. **For rare positives, PR-AUC is the honest curve.**

In [ ]:
fpr, tpr, roc_t = roc_curve(y_te, proba)
prec, rec, pr_t = precision_recall_curve(y_te, proba)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))
ax[0].plot(fpr, tpr, color="steelblue", lw=2.4,
           label=f"logistic (AUC = {roc_auc_score(y_te, proba):.4f})")
ax[0].plot([0, 1], [0, 1], "k--", lw=1.2, label="random (AUC = 0.5)")
ax[0].set_xlabel("false positive rate"); ax[0].set_ylabel("true positive rate (recall)")
ax[0].set_title("ROC curve"); ax[0].legend(fontsize=8)

ax[1].plot(rec, prec, color="crimson", lw=2.4,
           label=f"logistic (AP = {average_precision_score(y_te, proba):.4f})")
ax[1].axhline(y_te.mean(), color="black", ls="--", lw=1.2,
              label=f"random (= prevalence {y_te.mean():.3f})")
ax[1].set_xlabel("recall"); ax[1].set_ylabel("precision")
ax[1].set_title("Precision-recall curve"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"ROC-AUC          : {roc_auc_score(y_te, proba):.4f}")
print(f"Average precision: {average_precision_score(y_te, proba):.4f}")
print(f"Prevalence       : {y_te.mean():.4f}  <- the PR baseline")

In [ ]:
# Why PR-AUC matters when positives are rare
def compare_curves(prevalence, m_=20_000):
    yv = (rng.random(m_) < prevalence).astype(int)
    signal = rng.normal(yv * 1.1, 1.0)            # the same separability every time
    Xv = signal.reshape(-1, 1)
    Xa, Xb, ya, yb = train_test_split(Xv, yv, test_size=0.3, random_state=0, stratify=yv)
    mdl = LogisticRegression().fit(Xa, ya)
    pb = mdl.predict_proba(Xb)[:, 1]
    return roc_auc_score(yb, pb), average_precision_score(yb, pb), yb.mean()

print(f"{'prevalence':>12}{'ROC-AUC':>10}{'PR-AUC':>10}{'PR baseline':>14}")
for prev in (0.5, 0.2, 0.05, 0.01, 0.002):
    ra, pa, actual = compare_curves(prev)
    print(f"{actual:>12.4f}{ra:>10.4f}{pa:>10.4f}{actual:>14.4f}")
print("\nSeparability is identical in every row, yet PR-AUC collapses as positives get rare")
print("while ROC-AUC barely moves. With rare events, quote PR-AUC (and compare it against")
print("the prevalence baseline, not against 0.5).")

---
## 2.8 Regularisation and the `C` parameter

`LogisticRegression` is **regularised by default** — a fact that surprises everyone. The
objective is

$$\min_{\boldsymbol\beta}\ C\sum_{i}\text{log loss}_i + \tfrac{1}{2}\|\boldsymbol\beta\|^2$$

so `C` is the **inverse** of regularisation strength:

- **small `C`** → strong penalty → simpler model, coefficients shrunk toward 0
- **large `C`** → weak penalty → closer to unregularised maximum likelihood
- `penalty="l1"` (with `solver="liblinear"` or `"saga"`) gives sparse coefficients
- `penalty=None` turns it off entirely

Scale your features first, and tune `C` by cross-validation.

In [ ]:
Cs = np.logspace(-4, 4, 25)
tr_sc, te_sc, n_big = [], [], []
for C in Cs:
    p = Pipeline([("prep", ColumnTransformer([
                      ("num", StandardScaler(), num_cols),
                      ("cat", OneHotEncoder(drop="first"), cat_cols)])),
                  ("model", LogisticRegression(C=C, max_iter=5000))]).fit(X_tr, y_tr)
    tr_sc.append(log_loss(y_tr, p.predict_proba(X_tr)[:, 1]))
    te_sc.append(log_loss(y_te, p.predict_proba(X_te)[:, 1]))
    n_big.append(np.abs(p.named_steps["model"].coef_).sum())

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(Cs, tr_sc, "o-", color="steelblue", label="training log loss")
ax[0].plot(Cs, te_sc, "o-", color="crimson", label="test log loss")
ax[0].set_xscale("log"); ax[0].set_xlabel("C (larger = less regularisation)")
ax[0].set_ylabel("log loss"); ax[0].legend(fontsize=8)
ax[0].set_title("Under-regularised on the right, over-regularised on the left")
ax[1].plot(Cs, n_big, "o-", color="seagreen")
ax[1].set_xscale("log"); ax[1].set_xlabel("C"); ax[1].set_ylabel("sum of |coefficients|")
ax[1].set_title("Total coefficient size grows with C")
plt.tight_layout(); plt.show()

print(f"Best C by test log loss: {Cs[int(np.argmin(te_sc))]:.4f}")
print("With 3,000 rows and 6 features the curve is nearly flat -- regularisation matters")
print("far more when p is large relative to n.")

In [ ]:
# Tune C properly by cross-validation
gs = GridSearchCV(pipe, {"model__C": np.logspace(-3, 3, 25)},
                  cv=SKF, scoring="neg_log_loss").fit(X_tr, y_tr)
print(f"Best C          : {gs.best_params_['model__C']:.4f}")
print(f"Best CV log loss: {-gs.best_score_:.4f}")
print(f"Test log loss   : {log_loss(y_te, gs.predict_proba(X_te)[:, 1]):.4f}")
print(f"Test ROC-AUC    : {roc_auc_score(y_te, gs.predict_proba(X_te)[:, 1]):.4f}")

# L1 for sparsity
l1 = Pipeline([("prep", ColumnTransformer([
                   ("num", StandardScaler(), num_cols),
                   ("cat", OneHotEncoder(drop="first"), cat_cols)])),
               ("model", LogisticRegression(penalty="l1", C=0.05, solver="liblinear"))
              ]).fit(X_tr, y_tr)
names = l1.named_steps["prep"].get_feature_names_out()
print("\nL1 with C=0.05 keeps only:")
for nm, c in zip(names, l1.named_steps["model"].coef_[0]):
    if abs(c) > 1e-8:
        print(f"  {nm:<24} {c:+.4f}")

---
## 2.9 Interpreting coefficients as odds ratios

$e^{\beta_j}$ is the multiplicative change in the odds for a one-unit increase in $x_j$,
holding the others fixed.

| $e^\beta$ | Meaning |
|---|---|
| 2.0 | doubles the odds |
| 1.0 | no effect |
| 0.5 | halves the odds |

If you standardised the feature, "one unit" means "one standard deviation", which is usually
the more useful comparison. For categorical dummies, the odds ratio is relative to the
dropped reference level.

In [ ]:
best = gs.best_estimator_
feat_names = best.named_steps["prep"].get_feature_names_out()
coefs = best.named_steps["model"].coef_[0]

table = (pd.DataFrame({"feature": feat_names, "coefficient": coefs,
                       "odds_ratio": np.exp(coefs)})
         .assign(abs_c=lambda d: d.coefficient.abs())
         .sort_values("abs_c", ascending=False).drop(columns="abs_c"))
print(table.round(4).to_string(index=False))

plt.barh(table.feature[::-1], table.coefficient[::-1],
         color=["crimson" if c > 0 else "steelblue" for c in table.coefficient[::-1]])
plt.axvline(0, color="black", lw=1)
plt.xlabel("coefficient (log-odds, per standard deviation)")
plt.title("Red increases churn risk, blue decreases it")
plt.tight_layout(); plt.show()

print("\nHow to report it:")
for _, r in table.head(4).iterrows():
    direction = "increases" if r.coefficient > 0 else "decreases"
    print(f"  {r.feature}: {direction} the odds of churn by a factor of "
          f"{r.odds_ratio:.3f} per standard deviation")
print(f"\nMatches the simulation: long tenure and two-year contracts protect against churn;")
print("high monthly charges, support calls and fibre increase it.")

---
## 2.10 Class imbalance

When one class is rare, the loss is dominated by the majority and the model learns to predict
it almost always. Four responses, in order of preference:

1. **Do nothing to the data — change the threshold and the metric.** Often sufficient, and it
   keeps the probabilities honest.
2. **`class_weight="balanced"`** — reweight the loss so each class contributes equally.
   Cheap, principled, no data invented.
3. **Resampling** — undersample the majority or oversample the minority (SMOTE, from the
   `imbalanced-learn` package). Note that this **distorts the predicted probabilities**.
4. **Collect more minority examples.** Always best, rarely possible.

Never evaluate an imbalanced problem with accuracy.

In [ ]:
# A 3% positive rate
m_i = 6_000
Xi = rng.normal(size=(m_i, 6))
zi = -3.6 + 1.1*Xi[:, 0] + 0.8*Xi[:, 1] - 0.6*Xi[:, 2]
yi = (rng.random(m_i) < 1/(1+np.exp(-zi))).astype(int)
print(f"Positive rate: {yi.mean():.4f}\n")

Xa, Xb, ya, yb = train_test_split(Xi, yi, test_size=0.3, random_state=0, stratify=yi)

variants = {
    "majority-class baseline": DummyClassifier(strategy="most_frequent"),
    "plain logistic":          make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "class_weight=balanced":   make_pipeline(StandardScaler(),
                                             LogisticRegression(class_weight="balanced",
                                                                max_iter=1000)),
}
rows_i = []
for name, est in variants.items():
    est.fit(Xa, ya)
    pr = est.predict(Xb)
    try:
        pb = est.predict_proba(Xb)[:, 1]
        auc = roc_auc_score(yb, pb); ap = average_precision_score(yb, pb)
    except Exception:
        auc = ap = np.nan
    rows_i.append({"model": name,
                   "accuracy": accuracy_score(yb, pr),
                   "precision": precision_score(yb, pr, zero_division=0),
                   "recall": recall_score(yb, pr, zero_division=0),
                   "F1": f1_score(yb, pr, zero_division=0),
                   "ROC_AUC": auc, "PR_AUC": ap})
print(pd.DataFrame(rows_i).round(4).to_string(index=False))
print("\nThe baseline gets 97% accuracy by never predicting the positive class.")
print("class_weight='balanced' trades precision for a large gain in recall, and note that")
print("ROC-AUC and PR-AUC are UNCHANGED -- the ranking of customers is the same, only the")
print("implied threshold moved. That is why option 1 (move the threshold) usually suffices.")

---
## 2.11 Multiclass logistic regression

Two strategies:

- **Multinomial (softmax)** — one joint model, probabilities sum to 1 across classes:
  $$P(y=k\mid\mathbf{x}) = \frac{e^{\mathbf{x}^\top\boldsymbol\beta_k}}{\sum_{j}e^{\mathbf{x}^\top\boldsymbol\beta_j}}$$
- **One-vs-Rest (OvR)** — fit $K$ separate binary models, take the highest score.

`LogisticRegression` uses multinomial for multi-class problems with the default solvers, which
is usually what you want: the probabilities are coherent.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.multiclass import OneVsRestClassifier

wine = load_wine()
Xw, yw = wine.data, wine.target
Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(Xw, yw, test_size=0.3, random_state=0,
                                              stratify=yw)
print(f"{Xw.shape[0]} wines, {Xw.shape[1]} features, {len(wine.target_names)} classes: "
      f"{list(wine.target_names)}\n")

multi = make_pipeline(StandardScaler(),
                      LogisticRegression(max_iter=5000)).fit(Xw_tr, yw_tr)
ovr = make_pipeline(StandardScaler(),
                    OneVsRestClassifier(LogisticRegression(max_iter=5000))).fit(Xw_tr, yw_tr)

for name, mdl in [("multinomial (softmax)", multi), ("one-vs-rest", ovr)]:
    pr = mdl.predict(Xw_te)
    print(f"{name:<24} test accuracy {accuracy_score(yw_te, pr):.4f}, "
          f"log loss {log_loss(yw_te, mdl.predict_proba(Xw_te)):.4f}")

print("\nProbabilities for the first 5 test wines (multinomial):")
pp = pd.DataFrame(multi.predict_proba(Xw_te)[:5].round(4), columns=wine.target_names)
pp["row_sum"] = pp.sum(axis=1).round(4)
pp["true"] = [wine.target_names[i] for i in yw_te[:5]]
print(pp.to_string(index=False))

In [ ]:
ConfusionMatrixDisplay(confusion_matrix(yw_te, multi.predict(Xw_te)),
                       display_labels=wine.target_names).plot(cmap="Blues", colorbar=False)
plt.title("Multiclass confusion matrix (wine)")
plt.tight_layout(); plt.show()

print(classification_report(yw_te, multi.predict(Xw_te), target_names=wine.target_names))
print("With K classes you get one coefficient vector per class:")
print(f"  coefficient matrix shape = {multi[-1].coef_.shape}  (classes x features)")

---
## 2.12 Calibration: are the probabilities honest?

A model is **calibrated** if, among all cases it assigns probability 0.7, about 70% are
actually positive. This matters whenever you *use* the probability — for expected-value
calculations, for ranking by risk, for setting a cost-based threshold.

Logistic regression is naturally well calibrated because log loss is a **proper scoring
rule**: it is minimised only by the true probabilities. Models optimised for other objectives
(SVMs, boosted trees at high depth, naive Bayes) often are not, and need
`CalibratedClassifierCV`.

Measure it with a **reliability diagram** and the **Brier score**
$\frac{1}{n}\sum(\hat{p}_i - y_i)^2$ (lower is better).

In [ ]:
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

contenders = {
    "logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "naive Bayes":         GaussianNB(),
    "random forest":       RandomForestClassifier(n_estimators=200, random_state=0),
}
plt.plot([0, 1], [0, 1], "k--", lw=1.4, label="perfect calibration")
for name, est in contenders.items():
    est.fit(X_tr[num_cols], y_tr)
    pb = est.predict_proba(X_te[num_cols])[:, 1]
    frac_pos, mean_pred = calibration_curve(y_te, pb, n_bins=10, strategy="quantile")
    plt.plot(mean_pred, frac_pos, "o-", lw=2,
             label=f"{name} (Brier {brier_score_loss(y_te, pb):.4f})")
plt.xlabel("mean predicted probability"); plt.ylabel("observed fraction positive")
plt.title("Reliability diagram")
plt.legend(fontsize=8); plt.show()

print("Logistic regression tracks the diagonal closely. Naive Bayes is typically")
print("over-confident (pushed toward 0 and 1) because its independence assumption")
print("double-counts correlated evidence.")

In [ ]:
# Fixing a miscalibrated model
nb_raw = GaussianNB().fit(X_tr[num_cols], y_tr)
nb_cal = CalibratedClassifierCV(GaussianNB(), method="isotonic", cv=5).fit(X_tr[num_cols], y_tr)

for name, mdl in [("naive Bayes, raw", nb_raw), ("naive Bayes, calibrated", nb_cal)]:
    pb = mdl.predict_proba(X_te[num_cols])[:, 1]
    print(f"  {name:<26} Brier {brier_score_loss(y_te, pb):.4f}   "
          f"log loss {log_loss(y_te, pb):.4f}   ROC-AUC {roc_auc_score(y_te, pb):.4f}")
print("\nCalibration improves the PROBABILITIES without changing the RANKING much --")
print("notice ROC-AUC barely moves. Calibrate when you need the numbers, not just the order.")

---
## Exercises

**Exercise 1.** Build a logistic regression classifier for the breast cancer dataset. Report
accuracy, precision, recall, F1, ROC-AUC and PR-AUC against a baseline. Then choose the
threshold that guarantees at least 98% recall, and say what it costs in precision.

In [ ]:
# --- Solution 1 -------------------------------------------------------------
from sklearn.datasets import load_breast_cancer

bc = load_breast_cancer()
# sklearn codes malignant as 0; flip so that "malignant" is the positive class we care about
Xc, yc = bc.data, 1 - bc.target
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.3, random_state=0,
                                              stratify=yc)
print(f"{Xc.shape[0]} samples, {Xc.shape[1]} features, malignant rate {yc.mean():.3f}\n")

clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(Xc_tr, yc_tr)
pb_c = clf.predict_proba(Xc_te)[:, 1]
pr_c = clf.predict(Xc_te)

base = DummyClassifier(strategy="most_frequent").fit(Xc_tr, yc_tr)
print(f"Baseline accuracy : {base.score(Xc_te, yc_te):.4f}")
print(f"Accuracy          : {accuracy_score(yc_te, pr_c):.4f}")
print(f"Precision         : {precision_score(yc_te, pr_c):.4f}")
print(f"Recall            : {recall_score(yc_te, pr_c):.4f}")
print(f"F1                : {f1_score(yc_te, pr_c):.4f}")
print(f"ROC-AUC           : {roc_auc_score(yc_te, pb_c):.4f}")
print(f"PR-AUC            : {average_precision_score(yc_te, pb_c):.4f}")

# Threshold for at least 98% recall
ts = np.linspace(0.001, 0.999, 999)
ok = [(t, precision_score(yc_te, (pb_c >= t).astype(int), zero_division=0))
      for t in ts if recall_score(yc_te, (pb_c >= t).astype(int)) >= 0.98]
t_98, p_98 = max(ok, key=lambda z: z[1])
print(f"\nHighest threshold still giving >= 98% recall: {t_98:.3f}")
print(f"  recall    {recall_score(yc_te, (pb_c >= t_98).astype(int)):.4f}")
print(f"  precision {p_98:.4f}  (vs {precision_score(yc_te, pr_c):.4f} at threshold 0.5)")
print(f"  false alarms: {confusion_matrix(yc_te, (pb_c >= t_98).astype(int))[0,1]} "
      f"vs {confusion_matrix(yc_te, pr_c)[0,1]} at 0.5")
print("\nFor a screening test that is the right trade: a false alarm means a follow-up")
print("biopsy, a miss means an undetected malignancy.")

**Exercise 2.** Interpret a model. Fit logistic regression on the churn data with
standardised features, and write three sentences a non-technical manager could act on. Include
odds ratios.

In [ ]:
# --- Solution 2 -------------------------------------------------------------
final = Pipeline([
    ("prep", ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="first"), cat_cols)])),
    ("model", LogisticRegression(max_iter=2000)),
]).fit(X_all, y_all)

nm = final.named_steps["prep"].get_feature_names_out()
cf = final.named_steps["model"].coef_[0]
sd = df[num_cols].std()

out = pd.DataFrame({"feature": nm, "coef": cf, "odds_ratio": np.exp(cf)})
print(out.round(3).to_string(index=False))
print()
print("For a manager:")
tenure_or = np.exp(out.loc[out.feature == "num__tenure", "coef"].item())
print(f"  1. Tenure is the strongest protective factor: every extra "
      f"{sd['tenure']:.0f} months of")
print(f"     tenure multiplies the odds of churn by {tenure_or:.2f} -- roughly a "
      f"{(1-tenure_or)*100:.0f}% reduction.")
two_yr = np.exp(out.loc[out.feature == "cat__contract_two_year", "coef"].item())
print(f"  2. Moving a customer from a monthly to a two-year contract multiplies their")
print(f"     churn odds by {two_yr:.2f}. Contract migration is the highest-leverage action")
print(f"     we control directly.")
calls_or = np.exp(out.loc[out.feature == "num__support_calls", "coef"].item())
print(f"  3. Each additional {sd['support_calls']:.1f} support calls multiplies churn odds")
print(f"     by {calls_or:.2f}, so repeat callers should be routed to retention immediately.")
print()
print("Caveat to state alongside it: these are ASSOCIATIONS from observational data.")
print("Whether forcing customers onto two-year contracts actually reduces churn can only")
print("be established by an experiment.")

**Exercise 3.** Handle imbalance properly. Create a dataset with 1.5% positives, then compare
(a) plain logistic regression, (b) `class_weight="balanced"`, (c) plain model with a
cost-optimal threshold. Report which you would deploy given that a miss costs 50× a false
alarm.

In [ ]:
# --- Solution 3 -------------------------------------------------------------
m_x = 12_000
Xx = rng.normal(size=(m_x, 8))
zx = -4.6 + 1.2*Xx[:, 0] + 0.9*Xx[:, 1] - 0.7*Xx[:, 2] + 0.5*Xx[:, 3]
yx = (rng.random(m_x) < 1/(1+np.exp(-zx))).astype(int)
Xa, Xb, ya, yb = train_test_split(Xx, yx, test_size=0.3, random_state=0, stratify=yx)
print(f"Positive rate: {yx.mean():.4f} ({yx.sum()} positives)\n")

COST_MISS, COST_FA = 50.0, 1.0

plain = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xa, ya)
weighted = make_pipeline(StandardScaler(),
                         LogisticRegression(class_weight="balanced",
                                            max_iter=2000)).fit(Xa, ya)
pb_plain = plain.predict_proba(Xb)[:, 1]

def cost_of(y_true, y_pred):
    tn_, fp_, fn_, tp_ = confusion_matrix(y_true, y_pred).ravel()
    return fn_*COST_MISS + fp_*COST_FA, tp_, fp_, fn_

ts2 = np.linspace(0.0005, 0.5, 600)
costs2 = [cost_of(yb, (pb_plain >= t).astype(int))[0] for t in ts2]
t_opt = ts2[int(np.argmin(costs2))]

rows_x = []
for name, pred_ in [("(a) plain, threshold 0.5", plain.predict(Xb)),
                    ("(b) class_weight=balanced", weighted.predict(Xb)),
                    (f"(c) plain, threshold {t_opt:.4f}", (pb_plain >= t_opt).astype(int))]:
    c, tp_, fp_, fn_ = cost_of(yb, pred_)
    rows_x.append({"approach": name, "recall": recall_score(yb, pred_, zero_division=0),
                   "precision": precision_score(yb, pred_, zero_division=0),
                   "TP": tp_, "FP": fp_, "FN": fn_, "total_cost": c})
print(pd.DataFrame(rows_x).round(4).to_string(index=False))
print(f"\nTheory: the cost-optimal threshold is COST_FA/(COST_FA+COST_MISS) = "
      f"{COST_FA/(COST_FA+COST_MISS):.4f}")
print(f"Empirically we found {t_opt:.4f} -- close, as expected for a calibrated model.\n")
print("What I would deploy: (c). The plain model's probabilities are calibrated, so the")
print("threshold can be derived from the cost ratio directly and re-derived whenever the")
print("costs change -- no retraining. class_weight='balanced' bakes an implicit 1:65 cost")
print("ratio into the model and distorts the probabilities, which makes it harder to audit.")

**Exercise 4 (challenge).** A bank uses logistic regression for loan approvals and reports
"87% accuracy, ROC-AUC 0.91". Build a critique: what is missing, what could be wrong, and
what you would demand before signing off. Support each point with a computation.

In [ ]:
# --- Solution 4 -------------------------------------------------------------
m_l = 8_000
income = rng.lognormal(10.8, 0.5, m_l)
credit_score = rng.normal(680, 70, m_l).clip(300, 850)
existing_debt = rng.lognormal(9.5, 0.9, m_l)
group = rng.choice(["A", "B"], m_l, p=[0.75, 0.25])            # a protected attribute

# Default risk falls with income and credit score, rises with existing debt.
# Group B carries extra measured risk here only because of historical circumstance --
# exactly the situation a fairness audit has to disentangle.
z_default = (-1.9
             - 0.55*np.log(income / 50_000)
             - 0.011*(credit_score - 680)
             + 0.35*np.log(existing_debt / 12_000)
             + 0.45*(group == "B"))
default = (rng.random(m_l) < 1/(1+np.exp(-z_default))).astype(int)

loans = pd.DataFrame({"income": income, "credit_score": credit_score,
                      "existing_debt": existing_debt, "group": group, "default": default})
La, Lb = train_test_split(loans, test_size=0.3, random_state=0, stratify=loans.default)

feats_l = ["income", "credit_score", "existing_debt"]
mdl_l = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(
    La[feats_l], La.default)
pb_l = mdl_l.predict_proba(Lb[feats_l])[:, 1]
pr_l = mdl_l.predict(Lb[feats_l])

print("=== CRITIQUE ===\n")
print("1. NO BASELINE. Accuracy is meaningless without one.")
print(f"   default rate = {loans.default.mean():.4f}")
print(f"   'always predict no default' accuracy = "
      f"{DummyClassifier(strategy='most_frequent').fit(La[feats_l], La.default).score(Lb[feats_l], Lb.default):.4f}")
print(f"   model accuracy = {accuracy_score(Lb.default, pr_l):.4f}")
print("   -> most or all of the 'accuracy' is the base rate, not skill.\n")

print("2. WRONG METRIC for an imbalanced, cost-asymmetric decision.")
print(f"   precision {precision_score(Lb.default, pr_l, zero_division=0):.4f}, "
      f"recall {recall_score(Lb.default, pr_l, zero_division=0):.4f}")
print(f"   PR-AUC {average_precision_score(Lb.default, pb_l):.4f} vs prevalence "
      f"{Lb.default.mean():.4f}")
print("   -> ROC-AUC 0.91 with recall this low means the threshold is doing nothing useful.\n")

In [ ]:
print("3. NO COST MODEL. A default and a rejected good borrower cost different amounts.")
for cm_, cfa in [(20, 1), (5, 1), (1, 1)]:
    cs = [( (confusion_matrix(Lb.default, (pb_l >= t).astype(int)).ravel()[2]*cm_
            + confusion_matrix(Lb.default, (pb_l >= t).astype(int)).ravel()[1]*cfa), t)
          for t in np.linspace(0.02, 0.8, 40)]
    best_c, best_t = min(cs)
    print(f"   cost ratio {cm_}:{cfa} -> optimal threshold {best_t:.3f}")
print("   -> the threshold IS the policy, and nobody has stated the policy.\n")

print("4. NO CALIBRATION CHECK. Expected-loss pricing needs honest probabilities.")
print(f"   Brier score {brier_score_loss(Lb.default, pb_l):.4f}, "
      f"log loss {log_loss(Lb.default, pb_l):.4f}")
fp_, mp_ = calibration_curve(Lb.default, pb_l, n_bins=8, strategy="quantile")
print(f"   largest calibration gap across bins: {np.abs(fp_ - mp_).max():.4f}\n")

print("5. NO FAIRNESS AUDIT. The protected attribute is excluded, which is not enough")
print("   if the other features are correlated with it, or if the historical labels are biased.")
for g in ["A", "B"]:
    mask = (Lb.group == g).to_numpy()
    print(f"   group {g}: approval-rate proxy (predicted no-default) "
          f"{(pr_l[mask] == 0).mean():.4f}, "
          f"recall {recall_score(Lb.default[mask], pr_l[mask], zero_division=0):.4f}, "
          f"ROC-AUC {roc_auc_score(Lb.default[mask], pb_l[mask]):.4f}")
print("   -> report every metric per group. Equal overall AUC can hide unequal error rates.\n")

print("6. NO VALIDATION DETAIL. Demand: how was the split made? Is it time-ordered?")
print("   Were any features unavailable at application time? Was the test set used")
print("   for tuning? What is the CV spread, not just the point estimate?")
cv_auc = cross_val_score(mdl_l, La[feats_l], La.default, cv=SKF, scoring="roc_auc")
print(f"   CV ROC-AUC = {cv_auc.mean():.4f} +/- {cv_auc.std():.4f}\n")

print("=== WHAT I WOULD DEMAND BEFORE SIGN-OFF ===")
print("  * a stated cost model, and a threshold derived from it")
print("  * precision, recall and PR-AUC at that threshold, with confidence intervals")
print("  * a calibration curve and Brier score")
print("  * every metric broken down by protected group, plus a review of label bias")
print("  * a time-ordered validation (do not shuffle across the credit cycle)")
print("  * a feature audit for availability at decision time")
print("  * coefficients and odds ratios, because lending decisions must be explainable")
print("  * a monitoring plan: the population will drift")

---
## Summary

| Concept | Key point |
|---|---|
| Model | $P(y=1) = \sigma(\mathbf{x}^\top\boldsymbol\beta)$; linear in the **log-odds** |
| Loss | Log loss / cross-entropy; convex; punishes confident mistakes |
| Coefficients | $e^{\beta_j}$ = odds ratio; standardise to compare |
| Threshold | 0.5 is arbitrary — derive it from costs or a capacity constraint |
| Confusion matrix | TP, FP, FN, TN — every metric comes from these four numbers |
| Precision | Of those flagged, how many were real (false alarms are costly) |
| Recall | Of the real cases, how many caught (misses are costly) |
| ROC-AUC | Ranking quality; insensitive to prevalence |
| PR-AUC | The honest curve when positives are rare |
| `C` | **Inverse** regularisation strength; scale features first |
| Imbalance | Change the threshold and the metric before changing the data |
| Multiclass | Softmax (coherent probabilities) or one-vs-rest |
| Calibration | Logistic regression is naturally calibrated; check with a reliability diagram |

**Next up:** [Notebook 3 — Decision Trees](3.%20Decision%20Trees.ipynb), our first
non-linear model — and one you can read like a flowchart.